In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
from io import BytesIO
import requests

In [2]:
"""
STEP1 
Downloading the data from "agmanager" website and making sure 
our final dataframe has only the information we need (in terms of columns and rows)
"""
#start the Browser
driver = webdriver.Chrome()

try:
    url = "https://www.agmanager.info/"

    #open "agmanager" website
    driver.get(url)

    #move to "Grain Supply and Demand (WASDE)" page 
    wesda_page = driver.find_element(
            By.CSS_SELECTOR,
            "li[class='leaf menu-mlid-1183'] a"
        )
 
    wesda_page.click()

    #Press the button Download
    driver.find_element(By.CLASS_NAME, "btn-group").click()
    
    wait = WebDriverWait(driver, 10)

    #Press the text "Corn Supply and Demand Spreadsheet (WASDE)" to get the value of href
    link = wait.until(EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT, "Corn Supply and Demand Spreadsheet (WASDE)")))
    href = link.get_attribute("href")

    #Download the excel file and open it
    response = requests.get(href)
    response.raise_for_status()
    df = pd.read_excel(BytesIO(response.content), sheet_name='Annual Data', skiprows=9)

    #Create a list with the names of the columns we want to select
    column_names = ["Year","Planted Acres","Harvested Acres","Actual Yield","Production","Stocks",
                       "Exports","Imports","Total Supply","Total Usage","Ending Stocks"]

    #Create a new dataframe with just the selected columns
    df_selected_columns = df[column_names]

    #Create the final dataframe with the attribute "Year" > 1792 and "Year" < 2025
    df1 = df_selected_columns[(df_selected_columns["Year"] > 1972) & (df_selected_columns["Year"] < 2025)]
    
    #Reset index to start from 0
    df1 = df1.reset_index(drop = True)
    print(df1)
except:
    print("Something went wrong!")
finally:
    #close the driver
    driver.quit()

      Year Planted Acres Harvested Acres Actual Yield  Production     Stocks  \
0   1973.0         72253           62143    91.320451        5671        708   
1   1974.0         77935           65405    71.880734        4701        484   
2   1975.0         78583           67505    86.405325        5841        558   
3   1976.0         84374           71300    87.958042        6289        633   
4   1977.0         84328           71614    90.851955        6505       1136   
5   1978.0         81675           71930    101.08484        7268       1436   
6   1979.0         81394           72400   109.502762        7928       1710   
7   1980.0         84043           72961    90.945205        6639       2035   
8   1981.0         84097           74524   108.979866        8119       1392   
9   1982.0         81857           72719   113.273728        8235       2537   
10  1983.0         60217           51479    81.048544        4174       3523   
11  1984.0         80617           71897

In [ ]:
"""
STEP2 
Getting corn price data the data from "quickstats" website
"""
#start the Browser
driver = webdriver.Chrome()
url = "https://www.quickstats.nass.usda.gov/"

#open "agmanager" website  
driver.get(url)
wait = WebDriverWait(driver, 20)

#create a list containing the names of the dropdowns
header_names = ["sector_desc", "group_desc", "commodity_desc",
                "statisticcat_desc","short_desc","agg_level_desc"]

#create a list containing the values of the desired options
option_names = ["CROPS", "FIELD CROPS","CORN","PRICE RECEIVED",
                "CORN, GRAIN - PRICE RECEIVED, MEASURED IN $ / BU","NATIONAL"]

try: 
    
    #The following loop selects the options for the desired dropdown unitil "NATIONAL"
    for i in range(len(header_names)):   
        
        #We are going to use the number of records on the top right
        #of the screen to allow the program to wait until the page refreshes
        
        #Some operations are needed to turn the number of records into an integer
        #that we can actually use to define conditions
        number_of_records = driver.find_element(By.ID, "status_go").text.split()[0]
        number_of_records = int(number_of_records.replace(",",""))

        #Wait for the dropdown to be visible
        wait.until(EC.visibility_of_element_located((By.ID, header_names[i])))

        #Select the dropdown
        select_element = driver.find_element(By.ID, header_names[i])
        dropdown = Select(select_element)

        #select the option
        dropdown.select_by_value(option_names[i])

        #wait for the number of records to get smaller
        #which means the page refreshes
        wait.until(
            lambda d: int(
                d.find_element(By.ID, "status_go").text.split()[0].replace(",","")
            ) < number_of_records
        )

    #Create a list containing the names of the last two dropdowns
    header_names = ["year","freq_desc"]

    #create a list to select every year from 1973 to 2014
    option_names = ["1973"]
    while int(option_names[len(option_names) - 1]) < 2024 :
        option_names.append(str(int(option_names[len(option_names) - 1]) + 1))

    #with the following loop we select the years and the frequency of the data
    for i in range(len(header_names)):

        #Once again we are going to use the number of records
        number_of_records = int(driver.find_element(By.ID, "status_go").text.split()[0].replace(",",""))

        #wait fot the dropdown to be visible and click on the desired option
        wait.until(EC.visibility_of_element_located((By.ID, header_names[i])))
        select_element = driver.find_element(By.ID, header_names[i])
        dropdown = Select(select_element)

        if i == 0:
            
            #Select every year from 1973 to 2014
            for j in range(len(option_names)):
                dropdown.select_by_value(option_names[j])

        else:
            
            #Wait until the frequency dropdown is rebuilt after the page refreshes
            wait.until(EC.presence_of_element_located((By.ID, "freq_desc"))) 
            
            #Wait until MONTHLY actually exists as an option
            wait.until(
                    lambda d: any(
                        o.get_attribute("value") == "MONTHLY"
                        for o in d.find_elements(By.CSS_SELECTOR, "#freq_desc option")
                    )
            )

            #Re-fetch the select element           
            freq_dropdown = Select(driver.find_element(By.ID, "freq_desc"))
            
            #Select MONTHLY      
            freq_dropdown.select_by_value("MONTHLY")
            
            #Verify it stayed selected
            wait.until(
                lambda d: Select(
                   d.find_element(By.ID, "freq_desc")
                ).first_selected_option.get_attribute("value") == "MONTHLY"
            )

    #Re-apply NATIONAL because year/frequency resets it
    wait.until(EC.presence_of_element_located((By.ID, "agg_level_desc")))

    #verify that the option "NATIONAL" actually to be exist
    wait.until(
        lambda d: any(
            o.get_attribute("value") == "NATIONAL"
            for o in d.find_elements(By.CSS_SELECTOR, "#agg_level_desc option")
        )
    )

    #Select "NATIONAL" once again
    agg_select = Select(driver.find_element(By.ID, "agg_level_desc"))
    agg_select.select_by_value("NATIONAL")

    #Verify it stuck
    wait.until(
        lambda d: Select(
            d.find_element(By.ID, "agg_level_desc")
        ).first_selected_option.get_attribute("value") == "NATIONAL"
    )

    #click on "Get Data"
    driver.find_element(By.ID, "submit001_label").click()

    #Wait for the element "Printable" to appear and click on it 
    wait.until(EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT, "Printable")))
    driver.find_element(By.PARTIAL_LINK_TEXT, "Printable").click()

    #Get the html table
    tables = pd.read_html(driver.page_source)

    #Create the dataframe
    df2 = tables[0]
    
    print(df2)

except:
        print("Something went wrong!")
finally:
    #close the driver
    driver.quit()


In [8]:
"""
STEP3 
Getting crop progress condition data from "quickstats" website
"""
#start the Browser
driver = webdriver.Chrome()
url = "https://www.quickstats.nass.usda.gov/"

#open "agmanager" website  
driver.get(url)
wait = WebDriverWait(driver, 20)

#create a list containing the names of the dropdowns
header_names = ["sector_desc", "group_desc", "commodity_desc",
                "statisticcat_desc","agg_level_desc"]

#create a list containing the values of the desired options
option_names = ["CROPS", "FIELD CROPS","CORN","CONDITION","NATIONAL"]


try:        
    for i in range(len(header_names)): 
        
        #We are going to use the number of records on the top right
        #of the screen to allow the program to wait until the page refreshes
        
        #Some operations are needed to turn the number of records into an integer
        #that we can actually use to define conditions
        number_of_records = driver.find_element(By.ID, "status_go").text.split()[0]
        number_of_records = int(number_of_records.replace(",",""))

        #wait for the dropdown to be visible and select it
        wait.until(EC.visibility_of_element_located((By.ID, header_names[i])))
        select_element = driver.find_element(By.ID, header_names[i])
        dropdown = Select(select_element)
        
        #select the desired option
        dropdown.select_by_value(option_names[i])

        #wait for the number of records to decrease
        wait.until(
            lambda d: int(
                d.find_element(By.ID, "status_go").text.split()[0].replace(",","")
            ) < number_of_records
        )

    #create a list with the last two dropdowns
    header_names = ["year","freq_desc"]

    #create a list containing all years from 1986 to 2024
    option_names = ["1986"]
    while int(option_names[len(option_names) - 1]) < 2024 :
        option_names.append(str(int(option_names[len(option_names) - 1]) + 1))

    #the following loop is meant to select the years and the frequency of the data
    for i in range(len(header_names)):

        #we use the number of records
        number_of_records = int(driver.find_element(By.ID, "status_go").text.split()[0].replace(",",""))

        #wait for the dropdown to be visible and click on it
        wait.until(EC.visibility_of_element_located((By.ID, header_names[i])))
        select_element = driver.find_element(By.ID, header_names[i])
        dropdown = Select(select_element)
        
        if i == 0:
            # select every year from 1986 to 2024
            for j in range(len(option_names)):
                dropdown.select_by_value(option_names[j])

            wait.until(
                    lambda d: int(
                        d.find_element(By.ID, "status_go").text.split()[0].replace(",","")
                        ) < number_of_records
                )
        else:
            #Wait until the frequency dropdown is rebuilt
            wait.until(EC.presence_of_element_located((By.ID, "freq_desc")))
            #Wait until MONTHLY actually exists as an option
            wait.until(
                    lambda d: any(
                        o.get_attribute("value") == "WEEKLY"
                        for o in d.find_elements(By.CSS_SELECTOR, "#freq_desc option")
                    )
            )

            # Re-fetch the select element
            freq_dropdown = Select(driver.find_element(By.ID, "freq_desc"))
            #Select MONTHLY
            freq_dropdown.select_by_value("WEEKLY")

    #click on "Get Data"
    driver.find_element(By.ID, "submit001_label").click()

    #Wait for the element "Printable" to be available and click on it
    wait.until(EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT, "Printable")))
    driver.find_element(By.PARTIAL_LINK_TEXT, "Printable").click()

    #get the html table
    tables = pd.read_html(driver.page_source)
    #create the dataframe
    df3 = tables[0]
    print(df3)

except:
        print("Something went wrong!")
finally:
    #close the diver
    driver.quit()


     Program  Year    Period Week Ending Geo Level     State  State ANSI  \
0     SURVEY  2024  WEEK #22  2024-06-02  NATIONAL  US TOTAL         NaN   
1     SURVEY  2024  WEEK #22  2024-06-02  NATIONAL  US TOTAL         NaN   
2     SURVEY  2024  WEEK #22  2024-06-02  NATIONAL  US TOTAL         NaN   
3     SURVEY  2024  WEEK #22  2024-06-02  NATIONAL  US TOTAL         NaN   
4     SURVEY  2024  WEEK #22  2024-06-02  NATIONAL  US TOTAL         NaN   
...      ...   ...       ...         ...       ...       ...         ...   
4009  SURVEY  1986  WEEK #39  1986-09-28  NATIONAL  US TOTAL         NaN   
4010  SURVEY  1986  WEEK #39  1986-09-28  NATIONAL  US TOTAL         NaN   
4011  SURVEY  1986  WEEK #39  1986-09-28  NATIONAL  US TOTAL         NaN   
4012  SURVEY  1986  WEEK #39  1986-09-28  NATIONAL  US TOTAL         NaN   
4013  SURVEY  1986  WEEK #39  1986-09-28  NATIONAL  US TOTAL         NaN   

      Ag District  Ag District Code  County  ...  Zip Code  Region  \
0             NaN